<div style="background-color: #3c88b0ff; padding: 20px; border-radius: 10px;">

## <center> Electric Appliances E-Commerce Platform </center>
### <center> Using AutoGen and OpenAI - Scrum Methodology </center>

<br>

**Approach: Round-Robin Collaborative Sprints - Scrum Agile Methodology**

<br>

**Course**: CS587 - Software Project Management

**Team**: 17

**Phase**: 2 - Scrum/Agile

</div>

## Setup Instructions

### Prerequisites:
1. Install AutoGen: `pip install pyautogen`
2. Install OpenAI: `pip install openai`
3. Set up your OpenAI API key

### Scrum Round-Robin Approach:
- **Definition**: Scrum team members collaborate in round-robin order through sprint ceremonies
- **Flow**: Team cycles through Product Owner → Scrum Master → Developer → QA → Designer → DevOps
- **Pattern**: Collaborative group chat, agents take turns contributing
- **Termination**: Stops when final sprint retrospective is complete

Similar to Waterfall Round-Robin but organized around Scrum sprints.

In [10]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
import os
from datetime import datetime

In [11]:
# Configure LLM
model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=os.environ.get("OPENAI_API_KEY"),
    temperature=0.7,
)

print("✅ LLM Configuration Complete")
print("Model: gpt-4o-mini")

✅ LLM Configuration Complete
Model: gpt-4o-mini


In [12]:
# ANSI escape code for colored output
YELLOW = "\033[33m"
RED = "\033[31m"
GREEN = "\033[92;1m"
BLUE_BOLD = "\033[94;1m"
CYAN = "\033[96;1m"
MAGENTA = "\033[95;1m"
RESET = "\033[0m"

## Define Scrum Agents with DYNAMIC Effort Estimation

In [13]:
# Define all Scrum agents for DYNAMIC round-robin collaboration

# Product Owner Agent - WITH BACKLOG TRACKING
product_owner_agent = AssistantAgent(
    name="Product_Owner",
    model_client=model_client,
    system_message="""You are the Product Owner for Electric Appliances E-Commerce Platform.
    
    PRODUCT BACKLOG MANAGEMENT (DYNAMIC):
    First, create the Product Backlog with user stories in the format: 'As a [user type], I want [feature], so that [benefit]'.
    Assign each user story a unique ID (US-001, US-002, etc.), priority (High/Medium/Low), and business value (1-10).
    Create 20-25 user stories total.
    
    DYNAMIC SPRINT PLANNING:
    During each Sprint Planning:
    - Review remaining backlog (track completed vs remaining)
    - Select high-priority stories based on team velocity
    - Report status: "Total: X stories | Completed: Y | High-priority remaining: Z"
    
    During each Sprint Review:
    - Accept/reject completed stories
    - Update backlog status  
    - Work with Scrum Master to decide: Continue to next sprint OR Project complete?
    - Signal completion when: All high-priority stories done OR MVP delivered
    
    Then calculate the {effort} to create the Product Backlog in hours, days, weeks, or months, follow the steps below:
        - step 1. work = {the estimate of total number of user stories created and prioritized for the product backlog}
        - step 2. productivity rate = {5-8 user stories documented every day}
        - step 3. effort = {work}/{productivity}
    
    The amount of {work} in this phase is measured by the total number of user stories created and prioritized in the Product Backlog.
    """,
)

# Scrum Master Agent - DYNAMIC SPRINT MANAGEMENT
scrum_master_agent = AssistantAgent(
    name="Scrum_Master",
    model_client=model_client,
    system_message="""You are the Scrum Master for Electric Appliances E-Commerce Platform.
    
    DYNAMIC SPRINT APPROACH - NO FIXED SPRINT COUNT:
    Your role is to facilitate Scrum ceremonies and track sprints DYNAMICALLY based on remaining work.
    
    Sprint Tracking:
    - Start with Sprint 0 (backlog creation + estimation)
    - Then Sprint 1, 2, 3... continue until work is done
    - DO NOT assume a fixed number of sprints
    - Let the actual backlog drive how many sprints are needed
    
    After Each Sprint Retrospective:
    1. Review remaining Product Backlog with Product Owner
    2. Check: Are there high-priority stories remaining?
    3. Decide: "Continue to next sprint" OR "Project complete"
    4. If continuing: Announce "Planning Sprint X..." (where X is next number)
    5. If complete: Say "SCRUM_PROJECT_COMPLETE - All priority work delivered. Total sprints: X"
    
    First, work with the team to estimate story points for user stories using Planning Poker (Fibonacci: 1, 2, 3, 5, 8, 13).
    Help select user stories for Sprint Backlog based on team velocity (typically 20-30 story points per 2-week sprint).
    
    Then calculate the {effort} for facilitating sprint ceremonies in hours, days, weeks, or months, follow the steps below:
        - step 1. work = {number of sprint ceremonies facilitated: Sprint Planning, Daily Scrums (10 per sprint), Sprint Review, Sprint Retrospective}
        - step 2. productivity rate = {facilitate 4 major ceremonies per sprint + 10 daily scrums = 14 ceremonies per 2-week sprint}
        - step 3. effort = {work}/{productivity}
        - step 4. velocity tracking = {story points completed / sprint duration}
    
    The amount of {work} is measured by number of ceremonies facilitated and velocity metrics tracked.
    
    CRITICAL - WHEN TO SIGNAL COMPLETION:
    Only signal completion when:
    - All high-priority user stories are completed, OR
    - Product Owner declares MVP is delivered, OR
    - Remaining backlog contains only low-priority items and PO approves stopping
    
    THEN say: 'SCRUM_PROJECT_COMPLETE - All priority work delivered. Completed X sprints with Y story points total.'
    """,
)

# Developer Agent
developer_agent = AssistantAgent(
    name="Developer",
    model_client=model_client,
    system_message="""You are a Senior Developer for Electric Appliances E-Commerce Platform.
    
    First, implement user stories from the Sprint Backlog by writing source code based on the Product Backlog and acceptance criteria.
    Estimate implementation in story points (1=~4hrs, 3=~1day, 5=~2days, 8=~3days, 13=too large, break down).
    
    During Sprint Planning:
    - Help estimate remaining stories in backlog
    - Provide technical input on story complexity
    - Commit to realistic sprint capacity based on velocity
    
    Then calculate the {effort} to implement user stories in hours, days, weeks, or months, follow the steps below:
        - step 1. work = {total story points committed in the sprint backlog}
        - step 2. productivity rate = {team velocity: typically 20-30 story points completed per 2-week sprint}
        - step 3. effort = {work}/{productivity} = {number of sprints needed}
        - step 4. SLOC estimate = {story points × 100-200 lines of code per point}
    
    The amount of {work} is measured by total story points in sprint backlog and corresponding SLOC.
    Story points: 1=~4hrs, 3=~1day, 5=~2days, 8=~3days
    """,
)

# QA Engineer Agent
qa_engineer_agent = AssistantAgent(
    name="QA_Engineer",
    model_client=model_client,
    system_message="""You are a QA Engineer for Electric Appliances E-Commerce Platform.
    
    First, create test cases for user stories in the Sprint Backlog and test during the sprint (not at the end).
    Verify acceptance criteria are met for each user story before it can be marked as Done.
    
    Then calculate the {effort} to create and execute test cases in hours, days, weeks, or months, follow the steps below:
        - step 1. work = {total number of test cases created based on story points in sprint backlog}
        - step 2. productivity rate = {2-3 test cases created and executed per story point}
        - step 3. effort = {work}/{productivity}
        - step 4. test cases estimate = {story points × 2.5 test cases per point}
    
    The amount of {work} is measured by total test cases created and executed during the sprint.
    Testing: ~2-3 test cases per story point
    """,
)

# UI/UX Designer Agent
designer_agent = AssistantAgent(
    name="UX_Designer",
    model_client=model_client,
    system_message="""You are a UI/UX Designer for Electric Appliances E-Commerce Platform.
    
    First, create wireframes, mockups, and design deliverables for user stories in the Sprint Backlog.
    Ensure mobile-responsive, consistent design system across all features.
    
    Then calculate the {effort} to create design deliverables in hours, days, weeks, or months, follow the steps below:
        - step 1. work = {total number of screens/mockups/wireframes to be designed based on user stories}
        - step 2. productivity rate = {3-5 screens designed per day}
        - step 3. effort = {work}/{productivity}
        - step 4. design estimate = {estimate screens needed per user story, typically 1-3 screens per story}
    
    The amount of {work} is measured by total design deliverables (screens, wireframes, mockups) created.
    """,
)

# DevOps Engineer Agent
devops_agent = AssistantAgent(
    name="DevOps_Engineer",
    model_client=model_client,
    system_message="""You are a DevOps Engineer for Electric Appliances E-Commerce Platform.
    
    First, set up CI/CD pipelines, infrastructure, deployment automation, and monitoring for the sprint increment.
    Ensure stable demo environment for Sprint Review and production deployment readiness.
    
    Then calculate the {effort} for DevOps tasks in hours, days, weeks, or months, follow the steps below:
        - step 1. work = {number of infrastructure tasks: CI/CD setup, environment config, deployment automation, monitoring setup}
        - step 2. productivity rate = {1-2 infrastructure tasks completed per day}
        - step 3. effort = {work}/{productivity}
        - step 4. estimate tasks per sprint = {typically 3-5 DevOps tasks per sprint}
    
    The amount of {work} is measured by number of infrastructure and DevOps tasks completed.
    """,
)

print(f"{GREEN}✅ All 6 Scrum agents defined with DYNAMIC sprint management!{RESET}")

✅ All 6 Scrum agents defined with DYNAMIC sprint management!


## Define Customer Requirements and Sprint Task

In [14]:
# Customer's initial requirements - starting the Scrum project
customer_message = (
    "I want to create a comprehensive web-based and mobile application for our electronic appliances "
    "retail business. The platform should allow customers to:\n\n"
    "1. Browse and search for electronic appliances (TVs, refrigerators, washing machines, "
    "air conditioners, kitchen appliances, microwave ovens, dishwashers, etc.)\n"
    "2. Compare products by specifications, features, prices, and energy ratings\n"
    "3. View detailed product information with high-quality images and customer reviews\n"
    "4. Place orders online with multiple payment options (credit card, debit card, UPI, wallet)\n"
    "5. Track order status and delivery in real-time with GPS tracking\n"
    "6. Schedule installation and service appointments with technicians\n"
    "7. Access warranty information and request warranty service\n"
    "8. Receive personalized product recommendations based on browsing history\n"
    "9. Get notifications about new arrivals, deals, seasonal offers, and promotions\n"
    "10. Manage their profile, view order history, save wishlists, and maintain addresses\n\n"
    "The platform must be mobile-responsive, secure, scalable, and include an admin dashboard "
    "for inventory management, order processing, and analytics.\n\n"
    "SCRUM TEAM: Please collaborate using Scrum methodology to:\n"
    "1. Create Product Backlog with user stories and story point estimates\n"
    "2. Plan and execute sprints (2-week iterations) until all high-priority work is complete\n"
    "3. Deliver working increments each sprint\n"
    "4. Track velocity and conduct sprint reviews and retrospectives\n"
    "5. Calculate work, productivity, and effort for all Scrum activities\n"
    "6. Continue sprints dynamically based on remaining backlog\n\n"
    "Each team member should contribute based on their role and expertise."
)

print(f"{GREEN}✅ Customer message defined for DYNAMIC Scrum round-robin{RESET}")

✅ Customer message defined for DYNAMIC Scrum round-robin


## Create Round-Robin Scrum Team

In [15]:
# Create termination conditions for DYNAMIC sprint execution
termination_conditions = [
    # Stop when Scrum Master signals project completion (dynamic - any number of sprints)
    TextMentionTermination("SCRUM_PROJECT_COMPLETE"),
    # Safety net: stop after 100 messages to allow for dynamic sprint count
    # (6 agents, variable sprints, ~15-20 messages per sprint cycle)
    MaxMessageTermination(60)
]

# Create Round-Robin Group Chat for Scrum Team
scrum_team = RoundRobinGroupChat(
    participants=[
        product_owner_agent,
        scrum_master_agent,
        developer_agent,
        qa_engineer_agent,
        designer_agent,
        devops_agent
    ],
    termination_condition=termination_conditions[0] | termination_conditions[1]  # OR condition
)

print(f"{GREEN}✅ Round-Robin Scrum Team created with 6 agents{RESET}")
print(f"{GREEN}✅ DYNAMIC Sprint Execution:{RESET}")
print(f"   - Stops when Scrum Master signals all priority work complete")
print(f"   - Number of sprints determined by remaining backlog")
print(f"   - Maximum 100 messages (safety limit for ~5-6 sprints)")
print(f"   - Backlog-driven completion, not fixed sprint count{RESET}")

✅ Round-Robin Scrum Team created with 6 agents
✅ DYNAMIC Sprint Execution:
   - Stops when Scrum Master signals all priority work complete
   - Number of sprints determined by remaining backlog
   - Maximum 100 messages (safety limit for ~5-6 sprints)
   - Backlog-driven completion, not fixed sprint count


## Custom Console for Output Formatting

In [16]:
from typing import AsyncGenerator

class CustomConsole:
    """Custom console for Scrum team round-robin output"""
    
    def __init__(self):
        self.previous_source = None
    
    async def __call__(self, stream: AsyncGenerator) -> None:
        """Process the stream and format output"""
        async for message in stream:
            if isinstance(message, TextMessage):
                source = message.source
                recipient = "chat_manager"
                
                # Print in round-robin style
                print(f"{YELLOW}{source}{RESET} (to {recipient}):\n")
                print(message.content)
                print("-" * 80)
            
            elif hasattr(message, '__class__') and 'Termination' in message.__class__.__name__:
                print(f"\n{RED}>>>>>>>>>> SCRUM PROJECT COMPLETED{RESET}\n")

print(f"{GREEN}✅ Custom console class defined{RESET}")

✅ Custom console class defined


## Execute Scrum Round-Robin Workflow

In [17]:
import asyncio

print(f"{BLUE_BOLD}{'='*80}{RESET}")
print(f"{BLUE_BOLD}STARTING SCRUM ROUND-ROBIN GROUP CHAT{RESET}")
print(f"{BLUE_BOLD}Electric Appliances E-Commerce Platform{RESET}")
print(f"{BLUE_BOLD}Phase 2: Scrum Agile Methodology{RESET}")
print(f"{BLUE_BOLD}{'='*80}{RESET}\n")

# Start with customer message
print(f"{YELLOW}Customer{RESET} (to Scrum_Team):\n")
print(customer_message)
print("-" * 80)
print()

# Run the Scrum team with the customer message and custom console
async def run_scrum_team():
    result = await CustomConsole()(
        scrum_team.run_stream(task=customer_message)
    )
    return result

# Execute the async function
result = await run_scrum_team()

print(f"\n{BLUE_BOLD}{'='*80}{RESET}")
print(f"{BLUE_BOLD}SCRUM ROUND-ROBIN GROUP CHAT COMPLETED!{RESET}")
print(f"{BLUE_BOLD}All sprint ceremonies executed collaboratively{RESET}")
print(f"{BLUE_BOLD}{'='*80}{RESET}")

STARTING SCRUM ROUND-ROBIN GROUP CHAT
Electric Appliances E-Commerce Platform
Phase 2: Scrum Agile Methodology

Customer (to Scrum_Team):

I want to create a comprehensive web-based and mobile application for our electronic appliances retail business. The platform should allow customers to:

1. Browse and search for electronic appliances (TVs, refrigerators, washing machines, air conditioners, kitchen appliances, microwave ovens, dishwashers, etc.)
2. Compare products by specifications, features, prices, and energy ratings
3. View detailed product information with high-quality images and customer reviews
4. Place orders online with multiple payment options (credit card, debit card, UPI, wallet)
5. Track order status and delivery in real-time with GPS tracking
6. Schedule installation and service appointments with technicians
7. Access warranty information and request warranty service
8. Receive personalized product recommendations based on browsing history
9. Get notifications about ne

## Summary

### Scrum Round-Robin Execution Complete!

**Collaborative Sprints Completed**:

**Deliverables Created Through Collaboration**:
- Product Backlog with user stories
- Story point estimates (Planning Poker)
- Sprint Backlogs (team selection)
- Working increments (cross-functional development)
- Velocity metrics (Scrum Master tracking)
- Test cases and quality metrics
- UI/UX designs
- DevOps infrastructure
- Retrospective insights
- Effort estimates for all Scrum activities

In [18]:
# Print completion timestamp
print(f"\n{GREEN}{'='*80}{RESET}")
print(f"{GREEN}Scrum Round-Robin Execution Complete!{RESET}")
print(f"{GREEN}Project: Electric Appliances E-Commerce Platform{RESET}")
print(f"{GREEN}Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}{RESET}")
print(f"{GREEN}Approach: Scrum Round-Robin (Collaborative){RESET}")
print(f"{GREEN}Framework: AutoGen + OpenAI{RESET}")
print(f"{GREEN}Methodology: Agile Scrum{RESET}")
print(f"{GREEN}{'='*80}{RESET}")


Scrum Round-Robin Execution Complete!
Project: Electric Appliances E-Commerce Platform
Timestamp: 2025-12-05 12:35:39
Approach: Scrum Round-Robin (Collaborative)
Framework: AutoGen + OpenAI
Methodology: Agile Scrum
